# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RodionOm/Search-ranking-ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

## 1. Two paper findings + my methodology questions

**Finding A: "Refreshing mature content" (Finding #4).** 365+ day pages refreshed in the last 30 days show 3.2x health and 57x impressions vs untouched mature pages.
- *Label / outcome:* health score (a FlyRank composite that already includes impressions and position) and raw impressions.
- *Does the design carry the claim?* It is a comparison between two different groups of pages, not the same pages before and after a refresh. Teams usually choose to refresh pages that are already valuable, so part of the 57x may come from *which* pages were chosen (selection bias), not from the refresh itself. Also, health and impressions are not two independent confirmations, because health is partly built from impressions.
- *My questions:* How many pages are in the refreshed 365+ bucket? Is there a before/after view on the same pages, or a comparison matched by client and prior traffic?

**Finding B: "What predicts growth" (ML appendix).** Logistic regression separates growing vs declining pages with 71% holdout accuracy.
- *Label:* growing vs declining, taken from the 30-day trend (last 30 days vs previous 30 days).
- *Does the design carry the claim?* The holdout is a random 80/20 split, so pages from the same brand can sit in both train and test. The model can partly memorise brands. The base rate is not reported, so 71% cannot be compared with "always guess the majority class". "Recent impressions" is listed as a strong signal; if that window overlaps the trend window, the feature partly contains the answer.
- *My questions:* What is the majority-class accuracy? What is the score under a split grouped by brand? Do any feature windows overlap the label window?

The paper itself is careful: it calls the ML pages exploratory and says correlations are not causation. My questions are about how much weight the 71% and 57x numbers can carry, not about whether the direction is wrong. Sections 2 and 3 test both concerns on my own data.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    !git clone -q https://github.com/flyrank-bih/flyrank-ml-internship-starter.git
    %cd flyrank-ml-internship-starter

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
y = (df["trend_direction"] == "down").astype(int)   # label: 1 = page declined (last30 vs prev30)
groups = df["client_id"]

# the same 23 leakage-safe features as ML-05 / ML-08
X = pd.DataFrame(index=df.index)
for m in ["impressions", "clicks", "sessions"]:
    older = df[f"{m}_90d"] - df[f"{m}_last_30d"] - df[f"{m}_prev_30d"]
    X[f"log_{m}_older_30d"] = np.log1p(older)
    X[f"log_{m}_prev_30d"]  = np.log1p(df[f"{m}_prev_30d"])
X["early_momentum"] = X["log_impressions_prev_30d"] - X["log_impressions_older_30d"]
X["ctr_prev_30d"] = np.where(df["impressions_prev_30d"] > 0,
                             df["clicks_prev_30d"] / df["impressions_prev_30d"].replace(0, 1) * 100, 0)
X["content_age_days"] = df["content_age_days"]
X["has_word_count"] = df["word_count"].notna().astype(int)
X["word_count"] = df["word_count"].fillna(0)
X["has_keyword_data"] = df["search_volume"].notna().astype(int)
X["log_search_volume"] = np.log1p(df["search_volume"].fillna(0))
X["competition"] = df["competition"].fillna(0)
X["cpc"] = df["cpc"].fillna(0)
X = pd.concat([X, pd.get_dummies(df[["content_type", "main_intent"]].fillna("unknown"), dtype=int)], axis=1)
print("features:", X.shape, "| base rate:", round(y.mean(), 3))


# why base rate matters: accuracy of "always guess the majority class"
print("base rate (share of pages that declined):", round(y.mean(), 3))
print("majority-class accuracy (no model at all):", round(max(y.mean(), 1 - y.mean()), 3))

features: (30000, 23) | base rate: 0.542
base rate (share of pages that declined): 0.542
majority-class accuracy (no model at all): 0.542


## 2. My model under an honest split (before/after)
## 2. My model under an honest split (before/after)

Same 23 features, same models, two ways of splitting the data. Every score is out-of-fold, meaning each page is scored by a model that never trained on it.

| split | model | P@50 | P@500 | AUC |
|---|---|---|---|---|
| random KFold (rows shuffled) | logreg | 0.94 | 0.87 | 0.726 |
| random KFold (rows shuffled) | HGB | 0.92 | 0.93 | 0.790 |
| **GroupKFold by client** | logreg | **0.88** | **0.83** | **0.673** |
| **GroupKFold by client** | HGB | 0.62 | 0.69 | 0.698 |

Base rate = 0.542.

**What I observe:**
- With a random split, the same client's pages sit in both train and test, so every number goes up. That gain is memorisation of clients, not skill on new ones.
- The gap is largest for gradient boosting (P@500 drops from 0.93 to 0.69). A flexible tree model learns client-specific patterns that do not transfer.
- Logistic regression loses less (AUC drops from 0.726 to 0.673). It is too simple to memorise a client, so it generalises better. This explains why it beat HGB in ML-08.
- Under the honest split, logreg accuracy is 0.64 vs 0.542 for always guessing "declined": about 10 points of real skill. The paper's 71% (random split, no base rate shown) should be read with the same discount in mind.

**Honest number I report:** GroupKFold, logistic regression, P@50 = 0.88, AUC = 0.673.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupKFold, KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

def precision_at_k(scores, k):
    top = np.argsort(-np.asarray(scores), kind="stable")[:k]
    return y.iloc[top].mean()

def make_models():
    return {"logreg": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
            "hgb": HistGradientBoostingClassifier(random_state=42)}

def oof_scores(splitter):
    """Out-of-fold scores: every page is scored by a model that did not train on it."""
    oof = {"logreg": np.zeros(len(df)), "hgb": np.zeros(len(df))}
    for train_idx, test_idx in splitter:
        for name, model in make_models().items():
            model.fit(X.iloc[train_idx], y.iloc[train_idx])
            oof[name][test_idx] = model.predict_proba(X.iloc[test_idx])[:, 1]
    return oof

oof_random  = oof_scores(KFold(n_splits=5, shuffle=True, random_state=42).split(X))
oof_grouped = oof_scores(GroupKFold(n_splits=5).split(X, y, groups))

rows = []
for split_name, oof in [("random KFold (leaky)", oof_random), ("GroupKFold by client", oof_grouped)]:
    for model in ["logreg", "hgb"]:
        s = oof[model]
        rows.append([split_name, model, round(precision_at_k(s, 50), 2),
                     round(precision_at_k(s, 500), 2), round(roc_auc_score(y, s), 3),
                     round(((s > 0.5) == y).mean(), 3)])
print("base rate:", round(y.mean(), 3))
print(pd.DataFrame(rows, columns=["split", "model", "P@50", "P@500", "AUC", "accuracy"]).to_string(index=False))

base rate: 0.542
               split  model  P@50  P@500   AUC  accuracy
random KFold (leaky) logreg  0.94   0.87 0.726     0.665
random KFold (leaky)    hgb  0.86   0.93 0.792     0.720
GroupKFold by client logreg  0.88   0.83 0.673     0.640
GroupKFold by client    hgb  0.72   0.67 0.698     0.660


## 3. Leakage audit

## 3. Leakage audit

Timeline: features may only use **older30** (days 61-90) and **prev30** (days 31-60). The label is computed on **last30** (days 1-30). Any 90-day total contains last30, so it is banned.

**Checks on my final 23 features:**
- [x] **Timeline:** every feature is built from older30 or prev30 only. A name check confirms there are no last30, 90d-total or trend columns in X.
- [x] **No single feature towers:** the best single-feature AUC is 0.62 (log_impressions_prev_30d). Nothing "knows the answer" on its own.
- [x] **Trap test:** adding a leaky feature on purpose makes the score jump:

| feature set | AUC | P@50 |
|---|---|---|
| honest 23 features | 0.673 | 0.88 |
| + impressions_last_30d (label window) | **1.000** | 1.00 |
| + impressions_90d (contains label window) | **0.792** | 1.00 |

The harness catches leakage, so the honest 0.673 is believable. The 90d case matters most: one overlapping column adds about 0.12 AUC and makes the top-50 look perfect. This is exactly the risk behind "recent impressions" in Finding B of the paper.
- [x] **No product flags** (health score, tiers, the existing system's labels) used as features.
- [x] **Split grouped by client**, scores out-of-fold, base rate (0.542) printed next to every metric.
- **Limitation:** this is one 30-day snapshot, so a true time split (train on earlier months, test on later ones) is not possible with this data.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# (a) one feature alone: does any single column "know the answer"?
single = {c: roc_auc_score(y, X[c]) for c in X.columns}
single = pd.Series({c: max(a, 1 - a) for c, a in single.items()}).sort_values(ascending=False)
print("single-feature AUC (top 5):"); print(single.head(5).round(3).to_string())

# (b) trap test: add leaky features ON PURPOSE and watch the score jump
def grouped_scores(features):
    s = np.zeros(len(df))
    for tr, te in GroupKFold(n_splits=5).split(features, y, groups):
        m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(features.iloc[tr], y.iloc[tr])
        s[te] = m.predict_proba(features.iloc[te])[:, 1]
    return round(roc_auc_score(y, s), 3), round(precision_at_k(s, 50), 2)

trap_last = X.assign(LEAK_log_impressions_last_30d=np.log1p(df["impressions_last_30d"]))
trap_90d  = X.assign(LEAK_log_impressions_90d=np.log1p(df["impressions_90d"]))
for name, F in [("honest 23 features", X),
                ("+ impressions_last_30d (label window)", trap_last),
                ("+ impressions_90d (contains label window)", trap_90d)]:
    print(f"{name:45s} (AUC, P@50) = {grouped_scores(F)}")

# (c) name check: nothing from the label window in the final feature set
print("leaky columns in X:", [c for c in X.columns if "last_30d" in c or c.endswith("_90d") or "trend" in c])


single-feature AUC (top 5):
log_impressions_prev_30d     0.621
log_impressions_older_30d    0.612
content_age_days             0.591
word_count                   0.565
log_sessions_older_30d       0.552
honest 23 features                            (AUC, P@50) = (np.float64(0.673), np.float64(0.88))
+ impressions_last_30d (label window)         (AUC, P@50) = (np.float64(1.0), np.float64(1.0))
+ impressions_90d (contains label window)     (AUC, P@50) = (np.float64(0.792), np.float64(1.0))
leaky columns in X: []


## 4. Claim rewrite

## 4. Claim rewrite

**Bold version (what I was tempted to write):**
> "My model predicts which pages will lose traffic and beats the rule-based approach."

**Safe version:**
> On clients the model never saw (GroupKFold by client, 5 folds, out-of-fold scores), logistic regression ranked pages so that **44 of its top 50 were observed to decline** in the following 30 days (P@50 = 0.88), compared with 40 of 50 for the frozen ML-07 rule (0.80) and a base rate of 0.54. Overall separation is modest (AUC = 0.67), and P@50 varied between folds in ML-08. I read this as a **directional, decision-support ranking**: a short list for a human to review, not a forecast of how much traffic a page will lose.

**What changed and why:**
- "predicts" → "ranked … observed to decline": we measured a ranking against what happened, not a forecast.
- "beats" → exact numbers next to the rule and the base rate.
- Added the setup (unseen clients) and the weak spot (AUC 0.67, fold variation).
- Not claimed: causation. A page can fall in the list without a refresh helping it.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top50 = np.argsort(-oof_grouped["logreg"], kind="stable")[:50]
print("logreg top-50 on unseen clients: declined =", int(y.iloc[top50].sum()), "of 50")
print("P@50 =", round(precision_at_k(oof_grouped["logreg"], 50), 2),
      "| AUC =", round(roc_auc_score(y, oof_grouped["logreg"]), 3), "| base rate =", round(y.mean(), 3))

logreg top-50 on unseen clients: declined = 44 of 50
P@50 = 0.88 | AUC = 0.673 | base rate = 0.542


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.